# q k v Weights

In [1]:
from transformers import GPT2LMHeadModel

# Load GPT-2
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Pick first transformer block
attn = model.transformer.h[0].attn.c_attn

# Full weight and bias
print("Full c_attn weight:", attn.weight.shape)  # [2304, 768]
print("Full c_attn bias:", attn.bias.shape)      # [2304]

# Split into Q, K, V
hidden = model.config.hidden_size  # 768
w = attn.weight                    # [2304, 768]
b = attn.bias                      # [2304]

# Correct slicing along the first axis
q_w = w[0:hidden, :]               # [768, 768]
k_w = w[hidden:2*hidden, :]        # [768, 768]
v_w = w[2*hidden:3*hidden, :]      # [768, 768]

q_b = b[0:hidden]                  # [768]
k_b = b[hidden:2*hidden]           # [768]
v_b = b[2*hidden:3*hidden]         # [768]

print("Q weight:", q_w.shape, "bias:", q_b.shape)
print("K weight:", k_w.shape, "bias:", k_b.shape)
print("V weight:", v_w.shape, "bias:", v_b.shape)



Full c_attn weight: torch.Size([768, 2304])
Full c_attn bias: torch.Size([2304])
Q weight: torch.Size([768, 2304]) bias: torch.Size([768])
K weight: torch.Size([0, 2304]) bias: torch.Size([768])
V weight: torch.Size([0, 2304]) bias: torch.Size([768])


# GPT Tokenizer

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [4]:
model.eval()
text = "Rose loves sour fresh nuts like"

In [5]:
inputs = tokenizer(text, return_tensors="pt")

In [6]:
print (inputs)

{'input_ids': tensor([[31087, 10408, 11348,  4713, 14380,   588]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [7]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:", tokens)

Tokens: ['Rose', 'Ġloves', 'Ġsour', 'Ġfresh', 'Ġnuts', 'Ġlike']


In [8]:
decoded_text = tokenizer.decode(inputs["input_ids"][0])
print("Decoded text:", decoded_text)

Decoded text: Rose loves sour fresh nuts like


# Positional Embeddings

In [9]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load GPT2
tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Example text
text = "I love transformers"
x = tok(text, return_tensors="pt")

# Get position IDs (default is [0,1,2,...,seq_len-1])
pos_ids = torch.arange(x["input_ids"].size(1)).unsqueeze(0)

# Look up the positional embeddings
pos_emb = model.transformer.wpe(pos_ids)

print("Position IDs:", pos_ids)
print("Position Embeddings shape:", pos_emb.shape)   # [1, seq_len, hidden_size]
print("First position embedding vector:\n", pos_emb[0,0,:10])  # show first 10 dims


Position IDs: tensor([[0, 1, 2, 3]])
Position Embeddings shape: torch.Size([1, 4, 768])
First position embedding vector:
 tensor([-0.0188, -0.1974,  0.0040,  0.0113,  0.0638, -0.1050,  0.0369, -0.1680,
        -0.0491, -0.0565], grad_fn=<SliceBackward0>)


# GPT Final Input Token

In [10]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load GPT-2
tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

text = "I love transformers"
x = tok(text, return_tensors="pt")

# Token embeddings
tok_emb = model.transformer.wte(x["input_ids"])     # [batch, seq_len, hidden]
# Position embeddings
pos_ids = torch.arange(x["input_ids"].size(1)).unsqueeze(0)
pos_emb = model.transformer.wpe(pos_ids)            # [batch, seq_len, hidden]

# Final input embeddings (token + position)
input_emb = tok_emb + pos_emb
print("Shape:", input_emb.shape)  # [1, seq_len, hidden_size]

# Print each token + its input embedding
tokens = tok.convert_ids_to_tokens(x["input_ids"][0])
for i, token in enumerate(tokens):
    print(f"Token: {token:15s} | Input embedding (first 5 dims): {input_emb[0, i, :5]}")


Shape: torch.Size([1, 4, 768])
Token: I               | Input embedding (first 5 dims): tensor([ 0.1286, -0.2933,  0.1470, -0.0949,  0.0516], grad_fn=<SliceBackward0>)
Token: Ġlove           | Input embedding (first 5 dims): tensor([-0.1004, -0.1634, -0.1062, -0.0086, -0.0773], grad_fn=<SliceBackward0>)
Token: Ġtransform      | Input embedding (first 5 dims): tensor([ 0.0642, -0.2399,  0.2842,  0.1044, -0.1466], grad_fn=<SliceBackward0>)
Token: ers             | Input embedding (first 5 dims): tensor([-0.1840, -0.2391,  0.2144,  0.2043, -0.1209], grad_fn=<SliceBackward0>)


# GPT Attention Score

In [11]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load GPT-2
tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

text = "I love transformers"
x = tok(text, return_tensors="pt")

# Forward pass with attention outputs enabled
out = model(**x, output_attentions=True, return_dict=True)

# Attention is a tuple: one tensor per layer
attentions = out.attentions
print("Number of layers:", len(attentions))                 # 12 for gpt2 small
print("Shape of one attention head:", attentions[0].shape)  # [batch, heads, seq_len, seq_len]

# Example: attention scores from layer 0, head 0
print(attentions[0][0, 0])  # [seq_len, seq_len] matrix


`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Number of layers: 12
Shape of one attention head: torch.Size([1, 12, 4, 4])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.8994, 0.1006, 0.0000, 0.0000],
        [0.6229, 0.1545, 0.2226, 0.0000],
        [0.4932, 0.2957, 0.1584, 0.0528]], grad_fn=<SelectBackward0>)


# GPT Contexual Embedding 

In [12]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load GPT-2
tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

text = "I love transformers"
x = tok(text, return_tensors="pt")

# Forward pass with hidden states enabled
out = model(**x, output_hidden_states=True, return_dict=True)

# Last hidden state = contextual embeddings
contextual_emb = out.hidden_states[-1]   # [batch, seq_len, hidden_size]
print("Shape:", contextual_emb.shape)    # torch.Size([1, 3, 768])

# Print contextual embedding for each token
tokens = tok.convert_ids_to_tokens(x["input_ids"][0])
for i, token in enumerate(tokens):
    print(f"Token: {token:15s} | Embedding (first 5 dims): {contextual_emb[0, i, :5]}")


Shape: torch.Size([1, 4, 768])
Token: I               | Embedding (first 5 dims): tensor([-0.0796, -0.0654, -0.0842, -0.0337, -0.0758], grad_fn=<SliceBackward0>)
Token: Ġlove           | Embedding (first 5 dims): tensor([ 0.0757, -0.0351, -0.4318,  0.4180, -0.1469], grad_fn=<SliceBackward0>)
Token: Ġtransform      | Embedding (first 5 dims): tensor([ 0.0283, -0.4037, -0.7253,  0.3762,  0.0503], grad_fn=<SliceBackward0>)
Token: ers             | Embedding (first 5 dims): tensor([-0.0140, -0.1136, -1.8086, -0.3265, -0.0275], grad_fn=<SliceBackward0>)


# GPT Model Prediction

In [34]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
text = "Rose loves sour fresh fruits like strawberries,apples"
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [37]:
logits.shape

torch.Size([1, 8, 50257])

In [38]:
next_token_logits = logits[:, -1, :]

In [39]:
next_token_logits.shape

torch.Size([1, 50257])

In [40]:
next_token_logits

tensor([[ -98.4999, -101.2800, -103.2987,  ..., -107.5981, -103.5498,
         -100.3659]])

In [41]:
probs = torch.softmax(next_token_logits, dim=-1)


In [42]:
probs

tensor([[1.2918e-05, 8.0133e-07, 1.0643e-07,  ..., 1.4450e-09, 8.2804e-08,
         1.9989e-06]])

In [43]:
top_k = torch.topk(probs, k=5)

In [44]:
top_k

torch.return_types.topk(
values=tensor([[0.0727, 0.0572, 0.0424, 0.0418, 0.0366]]),
indices=tensor([[22514, 48389, 36973, 21382,   613]]))

In [45]:
for idx, score in zip(top_k.indices[0], top_k.values[0]):
    predicted_word = tokenizer.decode([idx])
    print(f"{predicted_word!r} : {float(score):.4f}")

' apples' : 0.0727
' oranges' : 0.0572
' strawberries' : 0.0424
' cher' : 0.0418
' pe' : 0.0366


# GPT Model Next word Predition - 2

In [46]:
model.eval()
text = "The kids are watching this movie with their parents and they want to watch the movie and see what"

In [47]:
inputs = tokenizer(text, return_tensors="pt")

In [48]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [49]:
next_token_logits = logits[:, -1, :]

In [50]:
probs = torch.softmax(next_token_logits, dim=-1)


In [51]:
top_k = torch.topk(probs, k=5)

In [52]:
for idx, score in zip(top_k.indices[0], top_k.values[0]):
    predicted_word = tokenizer.decode([idx])
    print(f"{predicted_word!r} : {float(score):.4f}")

' what' : 0.3129
' how' : 0.1451
' it' : 0.1199
' the' : 0.1009
' if' : 0.0673


# GPT Generate text

In [53]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

prompt = "Can you write a short beautiful love story?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(output[0], skip_special_tokens=True))


The story of a young man who has been a sailor for a long time. He is a sailor who is a good friend of his and he is a good friend of his. He is a good friend of his and he is a good friend of his. He is a good friend of his and he is a good friend of his. He is a good friend of his and he is 


# GPT - 5 Text Generation

In [54]:
from openai import OpenAI
import json

API_KEY = ""

client = OpenAI(api_key=API_KEY)

In [55]:

prompt = "Write a short bedtime story about a robot who learns to sing."

resp = client.responses.create(
    model="gpt-5",               # or "gpt-4.1", "gpt-4.1-mini"
    input=prompt,                # ✅ plain string is simplest
    max_output_tokens=800,       # give enough room for reasoning + visible text
    reasoning={"effort": "low"}, # optional: reduce reasoning token usage
)

# Most recent SDKs expose text here:
text = getattr(resp, "output_text", "")
print("\n--- Generated Output ---\n")
print(text.strip())



--- Generated Output ---

In a quiet workshop at the edge of a sleepy town, a small robot named Lumen ticked and whirred under the moonlight. Lumen had a polished chest that reflected the stars like tiny lanterns, and a heart made of careful gears that turned with patience. He could count the seconds, stack tiny screws into towers, and read every book aloud without stumbling. But when the evening breeze carried songs from the trees and houses, Lumen felt a warm ache in his wires. He wanted to sing.

He tried. He opened his speaker and let out a note that sounded like a kettle deciding to be brave. It wobbled, then squeaked, then cracked into a shy silence. Lumen dimmed his eyes and listened harder. Crickets rubbed the night into a rhythm. The river stitched silver thread through the reeds. From a window, a child hummed a bedtime tune, soft as a blanket being pulled up to a chin.

Lumen recorded it all. He replayed the crickets’ chirps and the water’s hush, noticing how they rose and f

# GPT -5 Top 5 Answers

In [15]:
prompt = "Write a short bedtime story about a robot who learns to sing."

print("\n--- Top 5 Possible Completions ---\n")

for i in range(1, 6):  # loop for 5 completions
    resp = client.responses.create(
        model="gpt-5",               # or "gpt-4.1", "gpt-4.1-mini"
        input=prompt,
        max_output_tokens=800,       # enough room for reasoning + visible text
        reasoning={"effort": "low"}, # optional: bias toward text
    )
    text = getattr(resp, "output_text", "").strip()
    print(f"[Option {i}]\n{text}\n")



--- Top 5 Possible Completions ---

[Option 1]
In a town that went to sleep softly each night, there lived a small robot named Lumo. Lumo was made of brushed silver and blinking lights, with a little round chest that ticked like a pocket watch. He could count stars and sort buttons and remember every story he ever heard. But there was one thing Lumo could not do.

He could not sing.

He tried, in the late afternoons when the laundry lines fluttered and the sun yawned its last warm yawn. Lumo would open the speaker in his chest and say, “La,” but it came out square and stiff, like a box that hadn’t been opened yet. He would read the sheet music he found in the library and measure the notes like footsteps, but none of it felt like a song.

“Maybe singing is just not in my software,” Lumo said to the crickets one evening, because crickets are very good listeners.

The crickets answered with their gentle chirr-chirr, which sounded a little like the tiniest violins practicing in the grass.

# Temperature

In [18]:
# pip install transformers torch --upgrade

from transformers import pipeline

# Create a text generation pipeline with GPT-2
generator = pipeline("text-generation", model="gpt2")

# Your prompt
prompt = "Once upon a time in a distant galaxy,"

# Generate text
outputs = generator(
    prompt,
    max_length=80,       # total tokens (prompt + generation)
    num_return_sequences=1,  # how many completions to return
    do_sample=True,      # use sampling (for diversity)
    top_p=0.95,
    temperature=0.9
)

# Print the result
print(outputs[0]["generated_text"])


Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time in a distant galaxy, the Human race had come to rule the galaxy. It was a mighty nation, a mighty kingdom. It was the one place where the Human race could rule. It was the place where the human race could become the greatest Galactic nation ever. It was the place where the human race could never get a foothold in the galaxy. It was the place where the
